<a href="https://colab.research.google.com/github/nov-cpu/8730-project/blob/API_Pulling/API_Pulling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 03-08-2026 (Version 6.0)
# ==========================================
# 1. INSTALL REQUIRED PACKAGES & IMPORTS
# ==========================================
!pip install google-search-results geopy pandas requests

import pandas as pd
import numpy as np
import requests
import sqlite3
import time
import os
import json # Ensure json is imported
from serpapi import GoogleSearch
from geopy.geocoders import Nominatim

# ==========================================
# 2. CONFIGURATION
# ==========================================
# Link directly to the raw .db file on GitHub
DB_URL = 'https://github.com/nov-cpu/8730-project/raw/API_Pulling/ev_infrastructure.db'
DB_FILE = 'ev_infrastructure.db'

API_KEY = '4d175cb9ae871aa6a0c485c3512c0293d51453c4c26a4fe6f8dd1393ed92fb34'
SEARCH_RADIUS_KM = 5.0
API_LIMIT = 5

geolocator = Nominatim(user_agent="uwindsor_ev_analytics_8730")

# ==========================================
# 3. RELATIONAL SQL DATABASE ETL PHASE
# ==========================================
print("📥 Downloading SQLite Database from GitHub...")
response = requests.get(DB_URL)
with open(DB_FILE, 'wb') as f:
    f.write(response.content)
print("✅ Database downloaded successfully.")

print("⚙️ Connecting to local SQLite Database engine...")
conn = sqlite3.connect(DB_FILE)

print("🔍 Executing SQL INNER JOIN query...")

# Query the pre-built tables from your .db file
sql_query = """
    SELECT
        s.[Station_ID  (PK)] AS id,
        s.Station_name AS station_name,
        s.Station_phone AS station_phone,
        l.Street_Address AS street_address,
        l.City AS city,
        l.State AS state,
        l.ZIP AS zip,
        l.Latitude AS latitude,
        l.Longitude AS longitude,
        s.Access_code AS access_code,
        s.Access_Detail_code AS access_detail_code,
        s.Access_Day_time AS access_days_time,
        s.EV_Pricing AS ev_pricing,
        s.Maximum_Vehicle_Class AS maximum_vehicle_class,
        s.Ev_Workplace_charging AS ev_workplace_charging,
        s.Restricted_access AS restricted_access,
        s.Date_last_confirmed AS date_last_confirmed
    FROM stations s
    INNER JOIN locations l
        ON s.[Location_ID (FK)] = l.[Location_ID (PK)]
"""

df_merged = pd.read_sql_query(sql_query, conn)
df_merged = df_merged.fillna("Unknown")
print(f"✅ SQL Pipeline Complete: {len(df_merged)} total records joined successfully via SQLite.")

# Close the database connection
conn.close()

# Save the merged SQL data as a flat CSV file for the React Dashboard
df_merged.to_csv('final_merged_dashboard_data.csv', index=False)
print("💾 CSV exported as 'final_merged_dashboard_data.csv'! Please download from the left folder menu.")

# ==========================================
# 4. HAVERSINE FORMULA (DISTANCE)
# ==========================================
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in KM
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
    return R * (2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)))

# ==========================================
# 5. USER INPUT & PROXIMITY SEARCH
# ==========================================
print("\n🌍 EV CHARGING STATION FINDER")
address = input("Enter a Canadian search location (e.g., 'Windsor, ON'): ")
location = geolocator.geocode(address)

if location:
    user_lat, user_lon = location.latitude, location.longitude
    print(f"\n📍 Target Coordinates Acquired: {user_lat:.4f}, {user_lon:.4f}")

    # Calculate distance for all stations in dataset
    df_merged['Distance_KM'] = calculate_distance(user_lat, user_lon, df_merged['latitude'], df_merged['longitude'])
    df_nearby = df_merged[df_merged['Distance_KM'] <= SEARCH_RADIUS_KM].sort_values(by='Distance_KM')

    if df_nearby.empty:
        print(f"⚠️ No stations found within {SEARCH_RADIUS_KM} km.")
    else:
        df_top = df_nearby.head(API_LIMIT)
        print(f"\nFound {len(df_nearby)} stations in {SEARCH_RADIUS_KM} km radius. Extracting live SerpAPI data for closest {API_LIMIT}...\n")

        # ==========================================
        # 6. DATA ENRICHMENT (SERPAPI SCRAPING)
        # ==========================================
        results_list = []
        for _, row in df_top.iterrows():
            station_name = row.get('station_name', 'Unknown')
            city = row.get('city', '')
            veh_class = row.get('maximum_vehicle_class', 'Standard Duty')

            params = {
                "engine": "google_maps",
                "q": f"EV Charging Station {station_name} {city}",
                "ll": f"@{row['latitude']},{row['longitude']},16z",
                "api_key": API_KEY
            }

            try:
                # 6.1 First API Call: Get Location Data & Data_ID
                search = GoogleSearch(params)
                results = search.get_dict()
                place = results.get("place_results", {})

                # Check local_results if place_results is empty
                if not place and "local_results" in results and len(results["local_results"]) > 0:
                     place = results["local_results"][0]

                rating = place.get("rating", "N/A")
                reviews_count = place.get("reviews", "N/A")
                data_id = place.get("data_id")

                print(f"✅ Scraped: {station_name} | Dist: {row['Distance_KM']:.2f} km | Rating: {rating} ⭐ ({reviews_count} total reviews) | Max Vehicle: {veh_class}")

                user_reviews = []

                # 6.2 Secondary API Call: Extract Individual Reviews if Data_ID exists
                if data_id:
                     try:
                         review_params = {
                             "engine": "google_maps_reviews",
                             "data_id": data_id,
                             "api_key": API_KEY,
                             "hl": "en",
                             "sort_by": "qualityScore"
                         }
                         review_search = GoogleSearch(review_params)
                         review_results = review_search.get_dict()
                         raw_reviews = review_results.get("reviews", [])

                         for rev in raw_reviews:
                              user_reviews.append({
                                  "Author": rev.get("user", {}).get("name", "Anonymous"),
                                  "Rating": rev.get("rating"),
                                  "Date": rev.get("date"),
                                  "Comment": rev.get("snippet", "No text provided")
                              })
                         print(f"  └─ Extracted {len(user_reviews)} individual reviews.")
                     except Exception as rev_err:
                         print(f"  └─ ❌ Error fetching reviews for {station_name}: {rev_err}")

                # Append everything to our results list
                extracted_data = {
                     "Station_Name": station_name,
                     "Distance_km": row['Distance_KM'],
                     "Rating": rating,
                     "Total_Reviews": reviews_count,
                     "Detailed_User_Reviews": user_reviews
                }
                results_list.append(extracted_data)

            except Exception as e:
                print(f"❌ API Error for {station_name}: {e}")
            time.sleep(1)

        # Optional: Save the structured JSON containing the reviews
        with open('station_detailed_reviews.json', 'w', encoding='utf-8') as f:
             json.dump(results_list, f, indent=4)
        print("\n💾 Nested JSON with reviews exported as 'station_detailed_reviews.json'")
else:
    print("❌ Location not found. Please try re-running with a valid address.")

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=e5b1b5efe756bac7ae19392a0c6a1508dfef2dc1db4948f9eb8407340b3131b5
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results
📥 Downloading SQLite Database from GitHub...
✅ Database downloaded successfully.
⚙️ Connecting to local SQLite Database engine...
🔍 Executing SQL INNER JOIN query...
✅ SQL Pipeline Complete: 15565 total records joined successfully via SQLite.
💾 CSV exported as 'final_merged_dashboard_data.csv'! Please download from the left folder menu.

🌍 EV CHARGING STATION FINDER
Enter a Canadian search location (e.g., 'Windsor, ON'): Toronto, ON

📍 Target Coordinates Acquired: 43.6535, -79.3839

Found 514 stations in 5.0 km radius. Extracting live SerpAPI data for closest 5...

✅ Scraped: COB RBCC 10 | Dist: 0.01 km | Rating: 3 ⭐ (3 tot